> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# 例 #1 - ストラドルの価格付け

もう少し複雑な（それでも単純な）演算を使うことで、ATMF ストラドルの価格を近似できます。

$$ STRADDLE_{ATMF} \approx \frac{2}{\sqrt{2\pi}} F \times \sigma \sqrt(T) $$
$$ \sigma = インプライド・ボラティリティ $$
$$ T = 満期までの期間 $$
$$ F = 原資産のフォワード $$

まず、ストラドルのインプライド・ボラティリティと満期までの時間（time-to-maturity）を定義することから始めましょう。なお、F は 1 に等しいと仮定し、ストラドルの価格はそれに応じてスケーリングできるものとします。 

In [ ]:
vol = 0.2
time = 1.

In [ ]:
2. * ( (1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

複数のストラドルを価格付けしたい場合にこれを何度も入力するのは、非常に面倒で、間違いも起きやすくなります。繰り返し使えるよう、この計算のための関数を定義しましょう

In [ ]:
def straddlePricer( vol, time ):
    return 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

これを実行しても、すぐには出力エリアに何も返ってこないことに注意してください。でも安心してください。関数は定義されており、使い始めることができます。以下では、関数の出力と、上のセルの出力とを比較できます。

In [ ]:
print( straddlePricer( 0.2, 1.0 ) )
print( 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) )
print( straddlePricer( 0.2, 1.0 ) == ( 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) ) )

関数に何を入力として使うかさえ伝えれば、入力の順序は問題になりません

In [ ]:
print( straddlePricer( time=1.0, vol=0.2 ) )
print( straddlePricer( vol=0.2, time=1.0 ) )

これは便利ですが、特定の入力をデフォルトにしたい場合はどうでしょうか。以下で初期の入力を設定することで、これらの引数を暗黙的に「オプション」（任意）にしています。まず、`time` だけをオプションの引数（入力）にします。  

In [ ]:
def straddlePricer( vol, time=1.0 ):
    return 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

In [ ]:
straddlePricer( 0.2 )

次に、`vol` と `time` の両方をオプションにします。

In [ ]:
def straddlePricer( vol=0.2, time=1.0 ):
    return 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

言い換えると、関数を呼び出す際にこれらの引数を渡す必要はありません。特に指定しない限り、`vol` には 0.2、`time` には 1.0 がデフォルトで使われます。

In [ ]:
straddlePricer()

In [ ]:
straddlePricer( 0.22 )

ストラドルの価格の式の分母には π があることに注意してください。ただし、ここまでで使った値（3.14）は大まかな近似です。もっと正確な値を使う方法はあるでしょうか。はい、`numpy` というライブラリを使うことができます。まず、以下でインポートしましょう。

In [ ]:
import numpy

`numpy` の関数には `numpy.xxxxx` と入力することでアクセスできます。ここで `xxxxx` は使いたい関数です。`numpy` における `pi` の実装は、単に `numpy.pi` です。

In [ ]:
numpy.pi

`numpy` と何度も入力するのは、かなり面倒になりがちです。名前を省略することで、もっと楽にしましょう。`numpy` の省略形に関する Python の慣例は `np` です。

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
np.pi

`numpy` には便利な平方根関数（`np.sqrt`）もあります

In [ ]:
np.sqrt( 4 )

`np.pi` と `np.sqrt` を私たちの単純なストラドルプライサーに組み込んで、もう少し正確で読みやすくしましょう。

In [ ]:
def straddlePricer( vol=0.2, time=1.0 ):
    return 2. * ( ( 1 / np.sqrt( 2 * np.pi ) ) * vol * np.sqrt( time ) )

straddlePricer()

元の実装と、新しく改良した実装との違いを見てみましょう。

In [ ]:
straddlePricer() - ( 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) )

この場合、精度と読みやすさの違いは大きくありませんが、その違いが貴重な場面もあります。上記の機能に加えて、`numpy` は他にも多くのことができます。例えば、乱数を生成できます。

In [ ]:
np.random.rand()

利用可能な関数を確認する方法はあるでしょうか。はい、`np.` の後にタブキーを押すだけです。

In [ ]:
#np.

あるいは、`np` に対して `dir` を呼び出して、何が含まれているかを確認することもできます。

In [ ]:
dir(np)

先ほどのストラドルの価格付けの例を続けると、モンテカルロ法を使ってストラドルを価格付けすることもできます。時間の経過に伴う資産の動きをシミュレーションするために、正規分布に従う乱数の集合を生成する必要があります。

In [ ]:
def straddlePricerMC(vol=0.2, time=1.0, mcPaths=100):
    dailyVol = vol / np.sqrt( 252. )
    resultSum = 0
    for p in range( mcPaths ):
        resultSum += np.abs( np.prod( ( 1 + np.random.normal( 0, dailyVol, int( round( time * 252 ) ) ) ) ) - 1 )
    return resultSum / mcPaths

straddlePricerMC()

ここでは新しいことがたくさん出てきています。1行ずつ紐解いていきましょう。

分散は時間に対して線形にスケーリングすることが分かっているので、次のどちらかを行うことができます

1. 分散を時間で割って平方根を取り、日次のボラティリティを得る、または
2. 分散の平方根（ボラティリティ）を取り、時間の平方根で割る
    
一般的に、私たちは通常ボラティリティの観点で考えるため、後者の方が明確で理解しやすいですが、どちらの方法を使っても構いません。

In [ ]:
# 上記の選択肢 #1
np.sqrt( vol ** 2 / 252 )

In [ ]:
# 2つの方法の比較
vol = 0.2
var = vol ** 2
sqrtVarOverTime = np.sqrt( var / 252 )
volOverSqrtTime = vol / np.sqrt( 252 )
valuesEqual = np.isclose( sqrtVarOverTime, volOverSqrtTime )
print( f'sqrtVarOverTime = {sqrtVarOverTime}\nvolOverSqrtTime = {volOverSqrtTime}\nAre they close? {valuesEqual}' )

次の行はそれほど面白いものではありませんが、累積和のデフォルト値を 0 に設定しています。つまり、`resultSum` を定義して 0 に設定しているだけです。これをしないとエラーになります。 

In [ ]:
resultSum = 0

次にループが出てきます。使えるループにはさまざまな種類があります。ここでは `for` ループを使います。これは「`range(mcPaths)` の各要素について繰り返す」という意味です。でも待ってください…`range(mcPaths)` とは何でしょうか。`range` は Python の組み込み関数で、0 から x-1 までの整数のリストに対するイテレータを返します。

In [ ]:
range10 = range( 10 )
lst = list( range10 )
print( lst )
print( len( lst ) )

私たちの場合、`p` を使って特に何かをしたいわけではないので、`_` に設定するのが良い習慣です。単にループを `mcPaths` 回繰り返したいだけです。デフォルトの場合、この関数はループを100回実行します。

In [ ]:
def straddlePricerMC( vol=0.2, time=1.0, mcPaths=100 ):
    dailyVol = vol / np.sqrt( 252. )
    resultSum = 0
    for _ in range( mcPaths ):
        resultSum += np.abs( np.prod( 1 + ( np.random.normal( 0, dailyVol, int( round( time * 252 ) ) ) ) ) - 1 )
    return resultSum / mcPaths

straddlePricerMC()

ループの各反復で関数が何をしているかを紐解くために、少しずつ見ていきましょう。最も内側の関数呼び出しから始めて、そこから逆向きにたどっていきます。`np.random.normal` メソッドが実際に何をするのか、ヘルプを見てみましょう。ありがたいことに、関数のドキュメントを見る便利な方法が2つあります。

1. help
2. ?

In [ ]:
help(np.random.normal)
# np.random.normal?

さて、help 関数によって、`np.random.normal` メソッドが3つのオプションの入力（平均、標準偏差、複数の乱数を生成するための配列のサイズ）を取ることが分かりました。デフォルトでは、平均 0、標準偏差 1 の分布となり、乱数を1つだけ返します。

In [ ]:
np.random.normal()

以下では、このメソッドを平均 0（ドリフトなし）、標準偏差を私たちの日次ボラティリティとして呼び出し、複数日分のリターンを生成します。具体的には、満期までの日数分のリターンを生成するよう依頼します。

In [ ]:
time = 1
nDays = time * 252
dailyVol = vol / np.sqrt( 252. )
print( nDays )

np.random.normal( 0, dailyVol, nDays )

さて、資産のリターンの時系列が得られたら、ストラドルはいくらになるでしょうか。私たちが関心を持っているのは資産の最終価格であり、ストラドルは ATM で組成されていると仮定できるため、資産の初期値（この場合は 1）からの乖離の絶対値を取るだけで済みます

In [ ]:
np.random.seed( 42 ) # 2つの乱数系列から同じ結果が得られるようにします

returns = np.random.normal( 0, dailyVol, time * 252 )
priceAtMaturity = np.prod( 1 + returns )
changeAtMaturity = priceAtMaturity - 1
absChangeAtMaturity = np.abs( changeAtMaturity )
print( absChangeAtMaturity )

# すべてを1行にまとめたもの
np.random.seed( 42 )
print( np.abs( np.prod( 1 + ( np.random.normal( 0, dailyVol, time * 252 ) ) ) - 1 ) )

上で行ったことをもう少し詳しく見てみましょう。今回は、私たちの作業を少し楽にするために、pandas と perspective という、さらに2つのライブラリを利用します。

In [ ]:
import pandas as pd
from perspective import psp

simulatedAsset = pd.DataFrame( np.random.normal( 0, dailyVol, time * 252 ) + 1, columns=['return'] )
simulatedAsset['price'] = ( 1 * simulatedAsset['return'] ).cumprod()
psp( simulatedAsset )

`for` ループは最終的には上記の処理を `mcPaths` 回行うだけであり、最終的にはパスの平均を取ることでストラドルの期待値を求めます。

In [ ]:
mcPaths = 100
resultSum = 0.
for _ in range(mcPaths):
    resultSum += np.abs( np.prod( 1 + np.random.normal( 0., dailyVol, time * 252 ) ) - 1 )
print( resultSum / mcPaths )

この価格は、元のプライサーによる価格にかなり近いです。パスを増やせば、さらに近づくはずです。

In [ ]:
straddlePricerMC(mcPaths=2000)

2000 本のパスは多いですが、それでも元の価格に収束していないようです。パスを増やせば、計算時間とのトレードオフが生じます。幸い、Jupyter を使えば、私たちの関数がどれくらい速いかを簡単に確認できます。

In [ ]:
%timeit straddlePricerMC(mcPaths=2000)

かなり速いですね。もっと多くのパスを実行できます。

In [ ]:
print(f"1 path: {straddlePricerMC(mcPaths=1)}")
print(f"2000 path: {straddlePricerMC(mcPaths=2000)}")
print(f"5000 path: {straddlePricerMC(mcPaths=5000)}")
print(f"10000 path: {straddlePricerMC(mcPaths=10000)}")
print(f"100000 path: {straddlePricerMC(mcPaths=100000)}")
print(f"Closed form approximation: {straddlePricer()}")

上記のモンテカルロの実装を改善できるでしょうか。もちろんできます。ランダムな資産系列を一度に生成できます。`np.random.normal` 関数の `size` 引数を思い出してください

In [ ]:
nDays = time * 252
size = (nDays, 15)
simulatedAsset = pd.DataFrame(np.random.normal(0, dailyVol, size))
simulatedAsset = (1 + simulatedAsset).cumprod()

simulatedAsset.tail()

いいですね！…matplotlib を使ってグラフに描き、可視化してみましょう。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')

fig = plt.figure(figsize=(8,6))
ax = plt.axes()
_ = ax.plot(simulatedAsset)

では、これをモンテカルロプライサーの `pandas` バージョンに組み込んでみましょう。

In [ ]:
def straddlePricerMCWithPD(vol=0.2, time=1, mcPaths=100000):
    dailyVol = vol / ( 252 ** 0.5 )
    randomPaths = pd.DataFrame( np.random.normal( 0, dailyVol, ( time*252, mcPaths ) ) )
    price = ( ( 1 + randomPaths ).prod() - 1 ).abs().sum() / mcPaths
    return price

straddlePricerMCWithPD()